In [7]:
import os
import sys
import json
import uuid
import datetime
import pandas as pd
from pathlib import Path

In [8]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done!")

In [22]:
from credit_risk.monitoring.prediction_logger import log_predictions
from credit_risk.api.schemas import RequestModel

In [23]:
uid = str(uuid.uuid4())
uid

'b901e443-ff6d-4f72-a441-e485bdaf47bc'

In [24]:
pred = 1
prob = 0.73
reason_codes = {
    "dti": 0.42,
    "fico_range_low": -0.31,
    "revol_util": 0.18,
    "annual_inc": -0.15,
    "term": 0.09
}

In [25]:
with open(project_root / "test_payload.json", 'r') as f:
    raw_input = json.load(f)

In [26]:
req = RequestModel(**raw_input)

In [27]:
issue_d = datetime.date.today()
issue_d

datetime.date(2026, 8, 16)

In [10]:
log_predictions(
    issue_d=issue_d,
    req=req,
    req_id=uid,
    pred=pred,
    prob=prob,
    reason_codes=reason_codes
)

2026-08-15 20:34:29.205 | INFO     | credit_risk.monitoring.prediction_logger:log_predictions:92 - Query executed successfully


In [16]:
# checking if this it got added in the database
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import psycopg2

In [ ]:
import os
import pandas as pd
verify_conn_str = ""

conn = psycopg2.connect(verify_conn_str)

sql = """
    SELECT * FROM prediction_logs
"""

df = pd.read_sql("SELECT * FROM prediction_logs ORDER BY logged_at DESC LIMIT 1", conn)

In [20]:
df

,id,request_id,logged_at,issue_d,loan_amnt,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,...,term,emp_length,home_ownership,verification_status,purpose,addr_state,initial_list_status,pred,prob,reason_codes
0,1,a297bcc0-7fd6-4dd3-87af-525b604c6f7b,2026-08-15 14:02:04.346050,2026-08-15,10000.0,310.0,120000.0,8.5,0.0,0.0,...,36 months,10+ years,MORTGAGE,Verified,credit_card,CA,w,1,0.73,"{'dti': 0.42, 'term': 0.09, 'annual_inc': -0.1..."


In [1]:
import time

t0 = time.time()
from credit_risk.monitoring.prediction_logger import pool
print("import + pool creation took:", time.time() - t0)

t0 = time.time()
conn = pool.getconn()
print("getconn took:", time.time() - t0)

cur = conn.cursor()
t0 = time.time()
cur.execute("SELECT 1")
print("query took:", time.time() - t0)

cur.close()
pool.putconn(conn)

2026-08-15 20:33:03.771 | INFO     | credit_risk.config:<module>:11 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


import + pool creation took: 2.7973692417144775
getconn took: 4.7206878662109375e-05
query took: 0.7439389228820801


In [20]:
conn_str = os.getenv("MONITORING_READER_DB_URI")
conn = psycopg2.connect(conn_str)

df = pd.read_sql("SELECT * FROM prediction_logs LIMIT 1;", conn)

conn.close()
df

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_54906/845238542.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM prediction_logs LIMIT 1;", conn)


,id,request_id,logged_at,issue_d,loan_amnt,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,...,term,emp_length,home_ownership,verification_status,purpose,addr_state,initial_list_status,pred,prob,reason_codes
0,1,a297bcc0-7fd6-4dd3-87af-525b604c6f7b,2026-08-15 14:02:04.346050,2026-08-15,10000.0,310.0,120000.0,8.5,0.0,0.0,...,36 months,10+ years,MORTGAGE,Verified,credit_card,CA,w,1,0.73,"{'dti': 0.42, 'term': 0.09, 'annual_inc': -0.1..."


In [28]:
conn_str = os.getenv("MONITORING_READER_DB_URI")
conn = psycopg2.connect(conn_str)
cur = conn.cursor()

cur.execute("INSERT INTO prediction_logs (request_id) VALUES ('00000000-0000-0000-0000-000000000000');")
conn.commit()

cur.close()
conn.close()

InsufficientPrivilege: permission denied for table prediction_logs


In [4]:
from credit_risk.dataset import AFTER_EDA, load_splits
from credit_risk.monitoring.log_loader import load_prediction_logs
from credit_risk.monitoring.psi import build_drift_report

2026-08-16 17:46:29.271 | INFO     | credit_risk.config:<module>:13 - PROJ_ROOT path is: /Users/ak007/SML/Credit-Risk-Default-Prediction-System


In [5]:
train_df, _, _, _ = load_splits(path=AFTER_EDA)
live_df = load_prediction_logs()

report = build_drift_report(reference_df=train_df, target_df=live_df, target_label='live')
report

2026-08-16 17:46:31.859 | INFO     | credit_risk.dataset:load_splits:261 | req_id=- - Checking if the files exists...
2026-08-16 17:46:31.867 | INFO     | credit_risk.dataset:load_splits:263 | req_id=- - Loading the Cached files...
2026-08-16 17:46:32.308 | INFO     | credit_risk.dataset:load_splits:271 | req_id=- - Loaded sucessfully all the splits and the metadata, Train_df shape: (466042, 110), val_df shape: (420204, 110), test_df shape: (431712, 110)
2026-08-16 17:46:32.333 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:18 | req_id=- - Starting to load the prediction logs from prediction_logs table...
2026-08-16 17:46:32.333 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:21 | req_id=- - Connecting to the DB...
2026-08-16 17:47:51.045 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:23 | req_id=- - Connection Successfull!
2026-08-16 17:47:51.047 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:26 | req_i

/Users/ak007/SML/Credit-Risk-Default-Prediction-System/credit_risk/monitoring/log_loader.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM prediction_logs", conn)


2026-08-16 17:48:00.075 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:32 | req_id=- - Read the logs successfully!
2026-08-16 17:48:00.076 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:37 | req_id=- - Adding the target place holder...
2026-08-16 17:48:00.076 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:41 | req_id=- - Correcting the Date dtype features...
2026-08-16 17:48:00.079 | INFO     | credit_risk.monitoring.log_loader:load_prediction_logs:45 | req_id=- - Correcting the potential null features dtypes...
2026-08-16 17:48:00.080 | INFO     | credit_risk.monitoring.psi:build_drift_report:154 | req_id=- - [build_drift_report] target_label='live': processing 58 numeric + 7 categorical features
2026-08-16 17:48:00.080 | INFO     | credit_risk.features:prep_one_split:214 | req_id=- - Inside Function: prep_one_split
2026-08-16 17:48:00.080 | INFO     | credit_risk.features:sorting_with_issue_d:140 | req_id=- - Sorting the 

,feature,PSI,drift_level,target_label
0,loan_amnt,0.015190,stable,live
1,installment,0.024982,stable,live
2,annual_inc,0.005923,stable,live
3,dti,0.113166,moderate,live
4,delinq_2yrs,0.011361,stable,live
...,...,...,...,...
60,home_ownership,0.005441,stable,live
61,verification_status,0.078367,stable,live
62,purpose,0.050574,stable,live
63,addr_state,0.114496,moderate,live


In [ ]:
from locustfile import row_to_payload

RuntimeError: cannot release un-acquired lock

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/ak007/SML/Credit-Risk-Default-Prediction-System/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, **kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ak007/SML/Credit-Risk-Default-Prediction-System/.venv/lib/python3.11/site-packages/ipykernel/subshell_manager.py", line 227, in _send_on_shell_channel
    assert current_thread().name == SHELL_CHANNEL_THREAD_NAME
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError
ERROR:tornado.general:Uncaught exception in zmqstream callback
Traceback (most recent call last):
  File "/Users/ak007/SML/Credit-Risk-Default-Prediction-System/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  File "/Users/ak007/SML/Credit-Risk-Default-Prediction-System/.venv/lib/python3.11/site-packages/